# 🏥 TriMedAgent V2 - Multi-turn Chatbot Demo (Orchestrator Pattern)

**Notebook này demo:**
- 🔄 **Multi-turn Conversation Memory** - Nhớ ngữ cảnh qua nhiều lượt hội thoại
- 🎯 **Full Pipeline**: Triage → Reasoning → Detection → Gatekeeper → Segmentation
- 📊 **Session Management**: Export/Import state để lưu trữ
- 💬 **Interactive Chat Interface**

---
## 1️⃣ Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Core imports
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import json
from datetime import datetime

# TriMedAgent Orchestrator V2 (Multi-turn Memory)
from src.trimed_orchestrator_v2 import (
    TriMedOrchestratorV2, 
    create_orchestrator,
    PipelineResult,
    ConversationState,
    ChatMessage
)

print("✅ Imports successful")
print(f"   Project Root: {PROJECT_ROOT}")
print(f"   Using: TriMedOrchestratorV2 (Multi-turn Memory)")

---
## 2️⃣ Initialize Orchestrator V2 (Multi-turn Memory)

In [ ]:
# Initialize Orchestrator V2 with Multi-turn Memory
orchestrator = create_orchestrator(
    session_id="demo_session",
    timeout=120,
    conv_template="llava_v1"
)

print("✅ Orchestrator V2 initialized with Multi-turn Memory")
print(f"\n📋 Session: {orchestrator.state.session_id}")
print(f"\n🔗 Worker URLs:")
for name, url in orchestrator.worker_urls.items():
    print(f"   • {name}: {url}")

In [ ]:
# Health Check - Verify workers are online
print("🔍 Checking worker connectivity...\n")

health = orchestrator.health_check()

for worker, is_online in health.items():
    status = "✅ Online" if is_online else "❌ Offline"
    print(f"  {worker}: {status}")

all_online = all(health.values())
if all_online:
    print("\n🎉 All workers are ready!")
else:
    print("\n⚠️ Some workers are offline. Pipeline may fail.")

---
## 3️⃣ Load Test Image

In [ ]:
# Load sample image
# Option 1: From local file
image_path = PROJECT_ROOT / "images" / "sample_xray.jpg"

# Option 2: Create placeholder if no sample exists
if not image_path.exists():
    print(f"⚠️ Sample image not found at: {image_path}")
    print("  Please provide an image path:")
    # Uncomment and set your image path:
    # image_path = Path("your/image/path.jpg")
    
    # Or download a sample:
    # !wget -O images/sample_xray.jpg "YOUR_URL"
else:
    image = Image.open(image_path)
    print(f"✓ Image loaded: {image_path.name}")
    print(f"  Size: {image.size}")
    print(f"  Mode: {image.mode}")
    
    # Display image
    plt.figure(figsize=(8, 8))
    plt.imshow(image, cmap='gray' if image.mode == 'L' else None)
    plt.title("Input Medical Image")
    plt.axis('off')
    plt.show()

---
## 4️⃣ Multi-turn Conversation Demo 💬

Sử dụng `orchestrator.chat()` để thực hiện hội thoại nhiều lượt.
Hệ thống sẽ nhớ ngữ cảnh từ các lượt trước.

In [ ]:
# Turn 1: Upload image and ask initial question
if 'image' in dir():
    print("💬 Turn 1: Initial question with image upload")
    print("=" * 60)
    
    response1, result1 = orchestrator.chat(
        user_input="What type of medical image is this? Describe what you see.",
        image=image
    )
    
    print(f"\n🤖 Assistant Response:\n{response1}")
    print(f"\n📊 Triage: {result1.triage.modality if result1.triage else 'N/A'}")
    print(f"⏱️ Time: {result1.execution_time:.2f}s")
    print(f"\n📝 Chat History: {len(orchestrator.state.messages)} messages")

In [ ]:
# Turn 2: Follow-up question (no image needed - uses previous context)
if 'image' in dir():
    print("💬 Turn 2: Follow-up question (remembers context)")
    print("=" * 60)
    
    response2, result2 = orchestrator.chat(
        user_input="Are there any abnormalities? Please describe them in detail."
    )
    
    print(f"\n🤖 Assistant Response:\n{response2}")
    print(f"\n📝 Chat History: {len(orchestrator.state.messages)} messages")
    print(f"📋 Conversation turns: {orchestrator.state.turn_count}")

---
## 5️⃣ Detection & Segmentation Turn 🔍

Khi query chứa action keywords (find, detect, segment...), pipeline đầy đủ sẽ chạy.

In [ ]:
# Turn 3: Detection query (triggers full pipeline)
if 'image' in dir():
    print("💬 Turn 3: Detection request (triggers Detect + Gatekeeper + Segment)")
    print("=" * 60)
    
    user_query = "Find and segment any tumors or abnormalities"
    
    response3, result3 = orchestrator.chat(
        user_input=user_query
    )
    
    print(f"\n🤖 Assistant Response:\n{response3}")
    
    # Show detection results
    print(f"\n📊 Detection Results:")
    print(f"   • Raw boxes: {len(result3.dino_raw_boxes)}")
    print(f"   • Verified boxes: {len(result3.verified_boxes)}")
    print(f"   • Masks: {len(result3.masks)}")
    print(f"\n📝 Chat History: {len(orchestrator.state.messages)} messages")

In [ ]:
# View full conversation history
print("📜 FULL CONVERSATION HISTORY")
print("=" * 60)

for i, msg in enumerate(orchestrator.state.messages):
    role_icon = "👤" if msg.role == "user" else "🤖"
    print(f"\n{role_icon} [{msg.role.upper()}] ({msg.timestamp[:19]})")
    print(f"   {msg.content[:200]}{'...' if len(msg.content) > 200 else ''}")

print("\n" + "=" * 60)
print(f"📊 Total turns: {orchestrator.state.turn_count}")
print(f"📊 Total messages: {len(orchestrator.state.messages)}")
print(f"📊 Images tracked: {len(orchestrator.state.images)}")

---
## 6️⃣ Visualization 📊

**Color Legend:**
- 🔴 **Red Dashed**: Raw DINO detections (before verification)
- 🟢 **Green Solid**: Verified boxes (after Gatekeeper)
- 🔵 **Blue Overlay**: MedSAM segmentation masks (α=0.5)

In [ ]:
def visualize_pipeline_result(
    image: Image.Image,
    result: PipelineResult,
    figsize: tuple = (16, 8),
    save_path: str = None
):
    """
    Visualize pipeline results with clear annotations.
    
    - Raw DINO boxes: Red dashed lines
    - Verified boxes: Green solid lines
    - Masks: Blue semi-transparent overlay
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Convert image to RGB if needed
    if image.mode == 'L':
        display_img = image.convert('RGB')
    else:
        display_img = image.copy()
    
    img_array = np.array(display_img)
    
    # ===== Left Panel: Raw Detections =====
    ax1 = axes[0]
    ax1.imshow(img_array)
    ax1.set_title("Raw DINO Detections", fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    # Draw raw boxes (red dashed)
    for i, box in enumerate(result.dino_raw_boxes):
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2,
            edgecolor='red',
            facecolor='none',
            linestyle='--'
        )
        ax1.add_patch(rect)
        
        # Add box number
        ax1.text(
            x1, y1 - 5, f"Box {i+1}",
            color='red',
            fontsize=10,
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    ax1.text(
        0.02, 0.98, f"Total: {len(result.dino_raw_boxes)} boxes",
        transform=ax1.transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )
    
    # ===== Right Panel: Verified Results =====
    ax2 = axes[1]
    ax2.imshow(img_array)
    ax2.set_title("Verified Results (After Gatekeeper)", fontsize=14, fontweight='bold')
    ax2.axis('off')
    
    # Draw raw boxes (faded red) for comparison
    for box in result.dino_raw_boxes:
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=1,
            edgecolor='red',
            facecolor='none',
            linestyle='--',
            alpha=0.3
        )
        ax2.add_patch(rect)
    
    # Draw verified boxes (green solid)
    for i, box in enumerate(result.verified_boxes):
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=3,
            edgecolor='limegreen',
            facecolor='none',
            linestyle='-'
        )
        ax2.add_patch(rect)
        
        # Add verified label
        ax2.text(
            x1, y1 - 5, f"✓ Verified {i+1}",
            color='green',
            fontsize=10,
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
    
    # Overlay masks (blue, semi-transparent)
    if result.masks:
        try:
            # Create mask overlay
            mask_overlay = np.zeros((*img_array.shape[:2], 4), dtype=np.float32)
            
            for mask in result.masks:
                # Handle different mask formats
                if isinstance(mask, str):
                    # Base64 encoded mask
                    import base64
                    from io import BytesIO
                    mask_bytes = base64.b64decode(mask)
                    mask_img = Image.open(BytesIO(mask_bytes))
                    mask_array = np.array(mask_img)
                elif isinstance(mask, np.ndarray):
                    mask_array = mask
                else:
                    continue
                
                # Resize mask to image size if needed
                if mask_array.shape[:2] != img_array.shape[:2]:
                    mask_pil = Image.fromarray(mask_array.astype(np.uint8))
                    mask_pil = mask_pil.resize(display_img.size, Image.NEAREST)
                    mask_array = np.array(mask_pil)
                
                # Apply blue color with transparency
                mask_bool = mask_array > 0
                if len(mask_bool.shape) > 2:
                    mask_bool = mask_bool[:, :, 0]
                
                mask_overlay[mask_bool, 0] = 0.0    # R
                mask_overlay[mask_bool, 1] = 0.5    # G
                mask_overlay[mask_bool, 2] = 1.0    # B
                mask_overlay[mask_bool, 3] = 0.5    # Alpha
            
            ax2.imshow(mask_overlay)
            
        except Exception as e:
            print(f"Warning: Could not overlay masks: {e}")
    
    # Stats text
    stats = f"Verified: {len(result.verified_boxes)}/{len(result.dino_raw_boxes)} boxes"
    if result.masks:
        stats += f"\nMasks: {len(result.masks)}"
    
    ax2.text(
        0.02, 0.98, stats,
        transform=ax2.transAxes,
        fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )
    
    # Add legend
    legend_elements = [
        Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Raw DINO Detection'),
        Line2D([0], [0], color='limegreen', linestyle='-', linewidth=3, label='Verified (Gatekeeper)'),
        patches.Patch(facecolor='royalblue', alpha=0.5, label='MedSAM Mask')
    ]
    fig.legend(
        handles=legend_elements,
        loc='lower center',
        ncol=3,
        fontsize=11,
        frameon=True,
        bbox_to_anchor=(0.5, -0.02)
    )
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Figure saved to: {save_path}")
    
    plt.show()
    
    return fig

In [ ]:
# Visualize results
if 'result' in dir() and 'image' in dir():
    print("📊 Generating Visualization...\n")
    
    fig = visualize_pipeline_result(
        image=image,
        result=result,
        figsize=(16, 8),
        save_path=None  # Set path to save: "output/result.png"
    )
else:
    print("⚠️ No results to visualize. Run the pipeline first.")

---
## 7️⃣ Detailed Gatekeeper Analysis

Examine which boxes were accepted/rejected and why.

In [ ]:
# Gatekeeper Analysis
if 'result' in dir() and result.gatekeeper_results:
    print("🔍 Gatekeeper Decision Analysis")
    print("=" * 60)
    
    for i, gk in enumerate(result.gatekeeper_results):
        status = "✅ VERIFIED" if gk.is_valid else "❌ REJECTED"
        
        print(f"\nBox {i+1}: {status}")
        print(f"  Coordinates: [{gk.box[0]:.0f}, {gk.box[1]:.0f}, {gk.box[2]:.0f}, {gk.box[3]:.0f}]")
        print(f"  Pathology Score: {gk.pathology_score:.1%}")
        print(f"  Normal Score: {gk.normal_score:.1%}")
        print(f"  Reason: {gk.reason}")
        
        # Visual bar
        path_bar = "█" * int(gk.pathology_score * 20)
        norm_bar = "░" * int(gk.normal_score * 20)
        print(f"  [Pathology] {path_bar}{norm_bar} [Normal]")
    
    print("\n" + "=" * 60)
    print(f"Summary: {len(result.verified_boxes)} verified, {len(result.rejected_boxes)} rejected")
    
else:
    print("No Gatekeeper results available.")

---
## 8️⃣ Session Management (Export/Import)

In [ ]:
# Export session for persistence
def save_session(orchestrator, filepath: str):
    """Save orchestrator session to JSON file."""
    session_data = orchestrator.export_session()
    
    # Convert to serializable format
    export_data = {
        "session_id": session_data.get("session_id"),
        "turn_count": session_data.get("turn_count"),
        "summary": session_data.get("summary", ""),
        "current_image_hash": session_data.get("current_image_hash"),
        "messages": [
            {
                "role": m["role"],
                "content": m["content"],
                "timestamp": m["timestamp"],
                "image_hash": m.get("image_hash")
            }
            for m in session_data.get("messages", [])
        ],
        "exported_at": datetime.now().isoformat()
    }
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Session saved to: {filepath}")
    return export_data

# Save current session
output_path = PROJECT_ROOT / "output" / "chat_session.json"
output_path.parent.mkdir(exist_ok=True)

if orchestrator.state and len(orchestrator.state.messages) > 0:
    exported = save_session(orchestrator, str(output_path))
    print(f"\n📄 Session Preview:")
    print(f"   • Session ID: {exported['session_id']}")
    print(f"   • Messages: {len(exported['messages'])}")
    print(f"   • Turns: {exported['turn_count']}")
else:
    print("⚠️ No session to export. Run chat first.")

In [ ]:
# Clear session and start fresh
print("🗑️ Clearing session...")
orchestrator.clear_session()
print(f"✅ Session cleared. Turn count: {orchestrator.state.turn_count}")

# Or start completely new session
new_session_id = orchestrator.start_session("new_demo_session")
print(f"✅ New session started: {new_session_id}")

---
## ✅ Complete!

Bạn đã sử dụng thành công **TriMedOrchestratorV2** với **Multi-turn Memory**.

### 🎯 Key Features:

| Feature | Description |
|---------|-------------|
| `orchestrator.chat()` | Main interface - tự động quản lý context |
| `orchestrator.state.messages` | Lịch sử hội thoại đầy đủ |
| `orchestrator.export_session()` | Export state để lưu trữ |
| `orchestrator.clear_session()` | Reset conversation |
| Auto-summarization | Tóm tắt history khi quá dài |

### 📚 Next Steps:
- Thêm RAG để tra cứu kiến thức y khoa: Xem `demo_trimedagent_rag.ipynb`
- Tùy chỉnh config trong `serve/labels.json`
- Thử với các loại ảnh y tế khác nhau